# 映前 (YingQian) — 多智能体电影推荐演示

基于 **HelloAgents** 的 Pipeline + Tool-use：

1. **画像 Agent**（无工具）→ TasteProfile  
2. **检索 Agent**（TMDB Tool）→ 真实候选片  
3. **推荐 Agent**（无工具）→ 白名单内精选 + 理由  

## 使用说明

1. 先配置 `backend/.env`（可从 `.env.example` 复制）  
2. 在项目根目录启动 Jupyter，按顺序运行本 Notebook  
3. 需能访问 TMDB 与你的 LLM API

---

## 第 1 部分：环境准备

In [ ]:
import os
import sys
from pathlib import Path

# 项目根目录 = 本 notebook 所在目录
ROOT = Path.cwd().resolve()
BACKEND = ROOT / "backend"
assert (BACKEND / "app").exists(), f"找不到 backend/app，请在项目根目录打开 notebook（当前: {ROOT}"

sys.path.insert(0, str(BACKEND))

# 优先加载 backend/.env
from dotenv import load_dotenv

env_path = BACKEND / ".env"
if not env_path.exists():
    raise FileNotFoundError(
        f"未找到 {env_path}\n"
        "请执行: copy .env.example backend\\.env  并填入 TMDB / LLM 密钥"
    )
load_dotenv(env_path)

print("ROOT   :", ROOT)
print("BACKEND:", BACKEND)
print("TMDB   :", "已配置" if (os.getenv("TMDB_ACCESS_TOKEN") or os.getenv("TMDB_API_KEY")) else "缺失")
print("LLM    :", "已配置" if os.getenv("LLM_API_KEY") else "缺失")
print("MODEL  :", os.getenv("LLM_MODEL_ID") or "(未设置)")

---

## 第 2 部分：构造推荐请求

In [ ]:
from app.models.schemas import RecommendRequest

request = RecommendRequest(
    mood="放松",
    party_type="独自",
    genres=["剧情", "喜剧"],
    max_runtime_minutes=120,
    region_preference="不限",
    year_preference="近10年",
    exclude_titles=[],
    spoilers_ok=False,
    free_text="不要太沉重，适合周末晚上",
    exclude_ids=[],
)

print(request.model_dump_json(indent=2, ensure_ascii=False))

---

## 第 3 部分：运行多智能体推荐流水线

> 完整一次大约 40–60 秒，请耐心等待。

In [ ]:
from app.agents.movie_recommender_agent import MultiAgentMovieRecommender

recommender = MultiAgentMovieRecommender()
result, trace_id = recommender.recommend(request)

print("trace_id:", trace_id)
print("fallback:", result.fallback)
if result.taste_profile:
    print("画像摘要:", result.taste_profile.summary)
    print("类型线索:", result.taste_profile.genre_hints)
print(f"推荐数量: {len(result.movies)}")
print("=" * 50)
for i, m in enumerate(result.movies, 1):
    print(f"{i}. {m.title} ({m.year or '?'})  评分={m.rating}")
    print(f"   理由: {m.reason}")
    print(f"   海报: {m.poster_url}")
    print()

---

## 第 4 部分（可选）：仅测 TMDB 连通性

In [ ]:
from app.services.movie_service import get_movie_service

svc = get_movie_service()
movies = svc.discover(with_genres="喜剧", sort_by="popularity.desc", page=1)
print(f"discover 返回 {len(movies)} 部，前 5 部：")
for m in movies[:5]:
    print(f"- {m.id} | {m.title} | {m.year} | {m.rating}")

---

## 总结

- 流水线：画像 → 检索(TMDB Tool) → 推荐(id 白名单)  
- Web 形态：`backend` FastAPI + `frontend` React  
- 若 TMDB 连接超时（WinError 10060），请检查代理/VPN 后重试